# VOOCKS Offline Worker Handoff Scaffold (DEMO)

This standard-library-only notebook validates a versioned manual job package, including input artifact bytes, then generates a deterministic synthetic result package. It does **not** connect to Colab, Drive, the network, GPUs, models, FFmpeg, or the Studio database. Worker identity and any T4 capability are DEMO / UNAUTHENTICATED assertions, not hardware verification.

No archives are accepted or extracted. Package paths must be direct files in the notebook working directory; symlinks, oversized files, unsafe artifact keys, malformed JSON, and checksum mismatches are rejected. No package content is executed.

In [ ]:
import base64
import hashlib
import hmac
import json
import os
import re
import stat
from datetime import datetime, timezone
from pathlib import Path

JOB_ID = re.compile(r"^[A-Za-z0-9][A-Za-z0-9._-]{0,63}$")
SHA256 = re.compile(r"^[a-f0-9]{64}$")
MAX_PACKAGE_BYTES = 16 * 1024 * 1024
MAX_INPUT_BYTES = 8 * 1024 * 1024
MAX_OUTPUT_BYTES = 8 * 1024 * 1024


def _unique_pairs(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError("Duplicate JSON field")
        result[key] = value
    return result


def _reject_constant(_value):
    raise ValueError("Invalid JSON constant")


def _safe_read(path):
    target = Path(path)
    if target.parent.resolve() != Path.cwd().resolve() or target.is_symlink():
        raise ValueError("Package path must be a direct, non-symlink file in this directory")
    descriptor = os.open(target, os.O_RDONLY | getattr(os, "O_NOFOLLOW", 0))
    try:
        metadata = os.fstat(descriptor)
        if not stat.S_ISREG(metadata.st_mode) or metadata.st_size > MAX_PACKAGE_BYTES:
            raise ValueError("Package must be a bounded regular file")
        chunks, total = [], 0
        while True:
            chunk = os.read(descriptor, min(65536, MAX_PACKAGE_BYTES + 1 - total))
            if not chunk:
                break
            chunks.append(chunk)
            total += len(chunk)
            if total > MAX_PACKAGE_BYTES:
                raise ValueError("Package exceeds the size limit")
        content = b"".join(chunks)
    finally:
        os.close(descriptor)
    if not content or len(content) > MAX_PACKAGE_BYTES:
        raise ValueError("Package is empty or exceeds the size limit")
    return content


def read_json(path="job_package.json"):
    try:
        value = json.loads(
            _safe_read(path).decode("utf-8"),
            object_pairs_hook=_unique_pairs,
            parse_constant=_reject_constant,
        )
    except (UnicodeDecodeError, json.JSONDecodeError) as error:
        raise ValueError("Package must contain valid UTF-8 JSON") from error
    if not isinstance(value, dict):
        raise ValueError("Package root must be an object")
    return value


def _timestamp(value):
    if not isinstance(value, str):
        raise ValueError("Invalid package timestamp")
    parsed = datetime.fromisoformat(value)
    if parsed.tzinfo is None or parsed.utcoffset() is None:
        raise ValueError("Package timestamps must include a timezone")
    return parsed


def _safe_artifact_ref(ref, *, output=False):
    fields = {"artifact_id", "backend", "key", "media_type", "max_bytes"} if output else {"artifact_id", "backend", "key", "sha256", "size_bytes", "media_type"}
    if not isinstance(ref, dict) or set(ref) != fields:
        raise ValueError("Invalid artifact reference fields")
    artifact_id, key = ref["artifact_id"], ref["key"]
    if not isinstance(artifact_id, str) or not JOB_ID.fullmatch(artifact_id):
        raise ValueError("Invalid artifact identifier")
    if (ref["backend"] != "local" or not isinstance(key, str) or not key or "\\" in key or key.startswith("/") or re.match(r"(?i)^https?://", key) or ":" in key or len(key) > 512 or any(part in {"", ".", ".."} for part in key.split("/")) or any(ord(char) < 32 for char in key)):
        raise ValueError("Unsafe artifact backend or key")
    if not isinstance(ref["media_type"], str) or "/" not in ref["media_type"]:
        raise ValueError("Invalid artifact media type")
    size_key = "max_bytes" if output else "size_bytes"
    size = ref[size_key]
    if type(size) is not int or size <= 0 or size > (4 * 1024**3 if output else MAX_INPUT_BYTES):
        raise ValueError("Invalid artifact size")
    if not output and (not isinstance(ref["sha256"], str) or not SHA256.fullmatch(ref["sha256"])):
        raise ValueError("Invalid input artifact SHA-256")


def validate_job_package(package, now=None):
    now = now or datetime.now(timezone.utc)
    if now.tzinfo is None or now.utcoffset() is None:
        raise ValueError("Validation time must include a timezone")
    expected = {"package_type", "schema_version", "manifest", "manifest_sha256", "input_artifacts", "authorization", "lease", "worker_profile", "worker_identity_authenticated", "exported_at", "execution_notice"}
    if not isinstance(package, dict) or set(package) != expected or package["package_type"] != "voocks.worker.job" or type(package["schema_version"]) is not int or package["schema_version"] != 2:
        raise ValueError("Unsupported or malformed job package")
    if package["execution_notice"] != "Offline handoff only. No hardware, worker identity, or execution environment is authenticated by this package." or package["worker_identity_authenticated"] is not False:
        raise ValueError("Execution notice is invalid")
    manifest = package["manifest"]
    manifest_fields = {"schema_version", "job_id", "idempotency_key", "project_id", "task_id", "workload", "input_artifacts", "output_artifacts", "resources", "max_attempts", "created_at", "deadline_at", "capabilities"}
    if not isinstance(manifest, dict) or set(manifest) != manifest_fields or type(manifest["schema_version"]) is not int or manifest["schema_version"] != 1:
        raise ValueError("Unsupported manifest schema")
    for key in ("job_id", "idempotency_key", "project_id", "task_id"):
        if not isinstance(manifest[key], str) or not JOB_ID.fullmatch(manifest[key]):
            raise ValueError("Invalid manifest identifier")
    canonical = json.dumps(manifest, sort_keys=True, separators=(",", ":"), ensure_ascii=True, allow_nan=False).encode("utf-8")
    if not isinstance(package["manifest_sha256"], str) or not SHA256.fullmatch(package["manifest_sha256"]) or not hmac.compare_digest(hashlib.sha256(canonical).hexdigest(), package["manifest_sha256"]):
        raise ValueError("Manifest SHA-256 mismatch")
    if manifest["workload"] not in {"cpu_ffmpeg", "gpu_inference"}:
        raise ValueError("Unsupported workload")
    resources = manifest["resources"]
    resource_fields = {"accelerator", "cpu_cores", "memory_mb", "min_vram_mb", "max_runtime_seconds", "max_output_bytes", "max_storage_bytes", "max_concurrent_jobs"}
    if not isinstance(resources, dict) or set(resources) != resource_fields:
        raise ValueError("Invalid resource declarations")
    for key, low, high in (("cpu_cores", 1, 64), ("memory_mb", 1, 262144), ("min_vram_mb", 0, 262144), ("max_runtime_seconds", 1, 86400), ("max_output_bytes", 1, 4 * 1024**3), ("max_storage_bytes", 1, 4 * 1024**3), ("max_concurrent_jobs", 1, 1), ("max_attempts", 1, 5)):
        value = manifest[key] if key == "max_attempts" else resources[key]
        if type(value) is not int or not low <= value <= high:
            raise ValueError("Resource or retry limit is outside policy")
    if type(resources["min_vram_mb"]) is not int or resources["accelerator"] not in {"none", "nvidia_t4"}:
        raise ValueError("Invalid accelerator requirement")
    if not isinstance(manifest["input_artifacts"], list) or not manifest["input_artifacts"] or not isinstance(manifest["output_artifacts"], list) or not manifest["output_artifacts"]:
        raise ValueError("Manifest must declare inputs and outputs")
    input_refs, output_refs = manifest["input_artifacts"], manifest["output_artifacts"]
    for ref in input_refs:
        _safe_artifact_ref(ref)
        if ref["key"] != f"studio-evidence/{ref['artifact_id']}.txt":
            raise ValueError("Input artifact is not associated with its declared evidence ID")
    for ref in output_refs:
        _safe_artifact_ref(ref, output=True)
        if ref["key"] != f"worker-jobs/{manifest['job_id']}/output.bin":
            raise ValueError("Output artifact is not associated with its declared job")
    if len({ref["artifact_id"] for ref in input_refs}) != len(input_refs) or len({ref["artifact_id"] for ref in output_refs}) != len(output_refs):
        raise ValueError("Duplicate artifact identifiers are not allowed")
    input_total = sum(ref["size_bytes"] for ref in input_refs)
    output_total = sum(ref["max_bytes"] for ref in output_refs)
    if input_total > min(MAX_INPUT_BYTES, resources["max_storage_bytes"]):
        raise ValueError("Input artifact total exceeds the handoff limit")
    if output_total > resources["max_output_bytes"]:
        raise ValueError("Output declarations exceed the handoff limit")
    if input_total + output_total > resources["max_storage_bytes"]:
        raise ValueError("Combined artifact storage exceeds the job limit")
    profile, auth, lease = package["worker_profile"], package["authorization"], package["lease"]
    if not all(isinstance(value, dict) for value in (profile, auth, lease)):
        raise ValueError("Profile, authorization, and lease must be objects")
    profile_fields = {"profile_version", "worker_id", "availability", "workloads", "cpu_cores", "memory_mb", "accelerators", "vram_mb", "processing_types", "operations", "language_tags", "model_ids", "expires_at", "is_synthetic", "provider_id"}
    if set(profile) != profile_fields or type(profile.get("profile_version")) is not int or profile["profile_version"] != 1:
        raise ValueError("Worker profile contract is invalid")
    if not isinstance(profile.get("worker_id"), str) or not JOB_ID.fullmatch(profile["worker_id"]) or profile["worker_id"] != lease.get("worker_id") or profile.get("is_synthetic") is not True or profile.get("availability") != "available":
        raise ValueError("Worker identity must be matching DEMO / UNAUTHENTICATED metadata")
    if not isinstance(profile.get("workloads"), list) or manifest["workload"] not in profile["workloads"] or type(profile.get("cpu_cores")) is not int or profile["cpu_cores"] < resources["cpu_cores"] or type(profile.get("memory_mb")) is not int or profile["memory_mb"] < resources["memory_mb"]:
        raise ValueError("Worker CPU, memory, or workload capability is insufficient")
    if (manifest["workload"] == "gpu_inference" and resources["accelerator"] != "nvidia_t4") or (manifest["workload"] == "cpu_ffmpeg" and resources["accelerator"] != "none"):
        raise ValueError("Workload and accelerator declarations conflict")
    if resources["accelerator"] != "none" and (resources["accelerator"] not in profile.get("accelerators", []) or type(profile.get("vram_mb")) is not int or profile["vram_mb"] < resources["min_vram_mb"]):
        raise ValueError("Worker accelerator/VRAM capability is insufficient")
    capabilities = manifest["capabilities"]
    capability_fields = {"operations", "language_tags", "model_ids"}
    if not isinstance(capabilities, dict) or set(capabilities) != capability_fields:
        raise ValueError("Manifest capability fields are invalid")
    for field in capability_fields:
        if not isinstance(capabilities[field], list) or not set(capabilities[field]).issubset(set(profile.get(field, []))):
            raise ValueError("Worker processing capability is insufficient")
    auth_fields = {"authorization_id", "director_id", "manifest_sha256", "authorized_at", "expires_at"}
    lease_fields = {"lease_id", "worker_id", "attempt", "acquired_at", "expires_at", "heartbeat_at"}
    if set(auth) != auth_fields or auth["manifest_sha256"] != package["manifest_sha256"] or not isinstance(auth.get("authorization_id"), str) or not JOB_ID.fullmatch(auth["authorization_id"]) or not isinstance(auth.get("director_id"), str) or not JOB_ID.fullmatch(auth["director_id"]):
        raise ValueError("Director authorization is missing or not manifest-bound")
    if set(lease) != lease_fields or not all(isinstance(lease[k], str) and JOB_ID.fullmatch(lease[k]) for k in ("lease_id", "worker_id")) or type(lease.get("attempt")) is not int or not 1 <= lease["attempt"] <= manifest["max_attempts"]:
        raise ValueError("Lease identity or attempt is invalid")
    acquired, expiry = _timestamp(lease["acquired_at"]), _timestamp(lease["expires_at"])
    heartbeat, auth_expiry = _timestamp(lease["heartbeat_at"]), _timestamp(auth["expires_at"])
    authorized = _timestamp(auth["authorized_at"])
    deadline, created = _timestamp(manifest["deadline_at"]), _timestamp(manifest["created_at"])
    if now >= expiry or now >= auth_expiry or now >= deadline or expiry > auth_expiry or expiry > deadline or authorized > now or authorized < created or acquired < authorized or heartbeat < acquired or heartbeat > now or heartbeat > expiry or expiry <= acquired or auth_expiry <= authorized:
        raise ValueError("Authorization, lease, or job deadline has expired")
    _timestamp(package["exported_at"])
    if package["worker_profile"].get("provider_id") is None or now >= _timestamp(profile["expires_at"]) or expiry > _timestamp(profile["expires_at"]):
        raise ValueError("Worker profile is expired or invalid")
    packaged_inputs = package["input_artifacts"]
    if not isinstance(packaged_inputs, list) or len(packaged_inputs) != len(input_refs):
        raise ValueError("Package must contain every verified input artifact")
    total = 0
    for item, reference in zip(packaged_inputs, input_refs):
        fields = {"artifact_id", "backend", "key", "sha256", "size_bytes", "media_type", "content_base64"}
        if not isinstance(item, dict) or set(item) != fields or any(item[key] != reference[key] for key in fields - {"content_base64"}):
            raise ValueError("Packaged input reference does not match manifest")
        encoded = item["content_base64"]
        if not isinstance(encoded, str) or len(encoded) > MAX_INPUT_BYTES * 4 // 3 + 8:
            raise ValueError("Input content exceeds the handoff limit")
        try:
            content = base64.b64decode(encoded, validate=True)
        except (ValueError, base64.binascii.Error) as error:
            raise ValueError("Input content is not canonical base64") from error
        if not content or len(content) != reference["size_bytes"] or base64.b64encode(content).decode("ascii") != encoded or not hmac.compare_digest(hashlib.sha256(content).hexdigest(), reference["sha256"]):
            raise ValueError("Input bytes do not match declared size or SHA-256")
        total += len(content)
    if total > min(MAX_INPUT_BYTES, resources["max_storage_bytes"]):
        raise ValueError("Input total exceeds the handoff limit")
    return package


def _write_json(path, package):
    target = Path(path)
    if target.parent.resolve() != Path.cwd().resolve() or target.exists() or target.is_symlink():
        raise ValueError("Refusing to overwrite or write outside the notebook directory")
    encoded = json.dumps(package, sort_keys=True, separators=(",", ":"), ensure_ascii=True, allow_nan=False).encode("utf-8")
    if not encoded or len(encoded) > MAX_PACKAGE_BYTES:
        raise ValueError("Result package is empty or exceeds the handoff limit")
    descriptor = os.open(target, os.O_WRONLY | os.O_CREAT | os.O_EXCL | getattr(os, "O_NOFOLLOW", 0), 0o600)
    try:
        with os.fdopen(descriptor, "wb") as output:
            output.write(encoded)
            output.flush()
            os.fsync(output.fileno())
    except OSError as error:
        target.unlink(missing_ok=True)
        raise ValueError("Result package could not be written safely") from error


def make_result_package(job_package, destination="result_package.json", now=None):
    package = validate_job_package(job_package, now=now)
    manifest, lease = package["manifest"], package["lease"]
    input_summaries = [{"artifact_id": item["artifact_id"], "sha256": item["sha256"], "size_bytes": item["size_bytes"]} for item in package["input_artifacts"]]
    outputs, total = [], 0
    for declaration in manifest["output_artifacts"]:
        demo_bytes = json.dumps({"label": "DEMO / UNAUTHENTICATED", "synthetic": True, "execution": "none", "job_id": manifest["job_id"], "task_id": manifest["task_id"], "project_id": manifest["project_id"], "workload": manifest["workload"], "input_artifacts": input_summaries, "manifest_sha256": package["manifest_sha256"]}, sort_keys=True, separators=(",", ":"), ensure_ascii=True).encode("utf-8")
        if len(demo_bytes) > min(declaration["max_bytes"], MAX_OUTPUT_BYTES):
            raise ValueError("Deterministic synthetic output exceeds its declared limit")
        total += len(demo_bytes)
        outputs.append({"artifact_id": declaration["artifact_id"], "backend": declaration["backend"], "key": declaration["key"], "sha256": hashlib.sha256(demo_bytes).hexdigest(), "size_bytes": len(demo_bytes), "media_type": declaration["media_type"], "content_base64": base64.b64encode(demo_bytes).decode("ascii")})
    if total > min(manifest["resources"]["max_output_bytes"], MAX_OUTPUT_BYTES):
        raise ValueError("Synthetic output total exceeds the job limit")
    completed = (now or datetime.now(timezone.utc)).astimezone(timezone.utc).isoformat()
    identity = f"{manifest['job_id']}|{lease['lease_id']}|{lease['attempt']}|{outputs[0]['sha256']}"
    result_id = "handoff-result-" + hashlib.sha256(identity.encode("utf-8")).hexdigest()[:32]
    result = {"package_type": "voocks.worker.result", "schema_version": 2, "status": "succeeded", "job_id": manifest["job_id"], "task_id": manifest["task_id"], "project_id": manifest["project_id"], "manifest_sha256": package["manifest_sha256"], "authorization_id": package["authorization"]["authorization_id"], "worker_id": lease["worker_id"], "lease_id": lease["lease_id"], "attempt": lease["attempt"], "completed_at": completed, "result_id": result_id, "is_synthetic": True, "worker_identity_authenticated": False, "error_code": None, "outputs": outputs}
    _write_json(destination, result)
    print("DEMO result package written; no workload was executed and worker identity is UNAUTHENTICATED.")
    return result


def make_interruption_package(job_package, destination="result_package.json", now=None):
    package = validate_job_package(job_package, now=now)
    manifest, lease = package["manifest"], package["lease"]
    completed = (now or datetime.now(timezone.utc)).astimezone(timezone.utc).isoformat()
    identity = f"{manifest['job_id']}|{lease['lease_id']}|{lease['attempt']}|interrupted"
    result = {"package_type": "voocks.worker.result", "schema_version": 2, "status": "interrupted", "job_id": manifest["job_id"], "task_id": manifest["task_id"], "project_id": manifest["project_id"], "manifest_sha256": package["manifest_sha256"], "authorization_id": package["authorization"]["authorization_id"], "worker_id": lease["worker_id"], "lease_id": lease["lease_id"], "attempt": lease["attempt"], "completed_at": completed, "result_id": "handoff-result-" + hashlib.sha256(identity.encode("utf-8")).hexdigest()[:32], "is_synthetic": True, "worker_identity_authenticated": False, "error_code": "WORKER_INTERRUPTED", "outputs": []}
    _write_json(destination, result)
    print("DEMO interruption report written; Director retry remains required.")
    return result


## Generate a deterministic synthetic result (DEMO)

This cell makes small inert JSON bytes derived from the job and verified input checksums. It does not process the input. The output and worker identity remain DEMO / UNAUTHENTICATED; the dashboard must recheck the lease and actual output bytes before queue acceptance.

In [ ]:
# Run after loading and validating job_package.json; no staged files or job commands are used.
# job = validate_job_package(read_json())
# result = make_result_package(job)